In [0]:
USE databricks_wanderbricks_dataset_dais_2025.wanderbricks;

-- 1. Total amenities
SELECT COUNT(*) AS total_amenities 
FROM amenities;

-- 2. What amenity categories exist
SELECT DISTINCT category 
FROM amenities;

-- 3. Most common amenities
SELECT 
    a.amenity_id,
    a.name AS amenity_name,
    a.category,
    COUNT(pa.property_id) AS property_count
FROM property_amenities pa
JOIN amenities a 
    ON pa.amenity_id = a.amenity_id
GROUP BY a.amenity_id, a.name, a.category
ORDER BY property_count DESC;

-- 4. How many amenities does each property have
SELECT 
    property_id,
    COUNT(amenity_id) AS total_amenities
FROM property_amenities
GROUP BY property_id
ORDER BY total_amenities DESC;

-- 5. Avg amenities per property
WITH property_amenity_counts AS (
    SELECT 
        p.property_id,
        COUNT(pa.amenity_id) AS total_amenities
    FROM properties p
    LEFT JOIN property_amenities pa 
        ON p.property_id = pa.property_id
    GROUP BY p.property_id
)
SELECT AVG(total_amenities) AS avg_amenities_per_property
FROM property_amenity_counts;

-- 6. Most common amenities within each country (Top ranked per country using CTE)
WITH ranked_amenities AS (
    SELECT 
        c.country, 
        a.name AS amenity, 
        COUNT(*) AS amenity_count,
        ROW_NUMBER() OVER(PARTITION BY c.country ORDER BY COUNT(*) DESC) AS rank_no
    FROM property_amenities pa
    JOIN amenities a 
        ON pa.amenity_id = a.amenity_id
    JOIN properties p 
        ON pa.property_id = p.property_id
    JOIN cities c 
        ON p.city_id = c.city_id
    GROUP BY c.country, a.name
)
SELECT country, amenity, amenity_count, rank_no
FROM ranked_amenities
WHERE rank_no <= 3
ORDER BY country, rank_no;

-- 7. Avg property price by amenity category
SELECT 
    a.category,
    AVG(p.base_price) AS avg_property_price,
    COUNT(DISTINCT p.property_id) AS property_count
FROM properties p
JOIN property_amenities pa 
    ON p.property_id = pa.property_id
JOIN amenities a 
    ON pa.amenity_id = a.amenity_id
GROUP BY a.category
ORDER BY avg_property_price DESC;